# OpenMP with Flang REPL

These are ordinary interactive Fortran cells. The hosted kernel starts Flang with `-fopenmp`, preloads `libomp`, and captures only the execution phase so threaded compilation and output forwarding do not interfere with each other.

In [ ]:
integer :: thread_count
thread_count = 0
!$omp parallel
!$omp atomic update
thread_count = thread_count + 1
!$omp end atomic
!$omp end parallel
print *, 'OpenMP threads:', thread_count

In [ ]:
subroutine reduction_demo()
  integer(kind=8) :: total, value
  total = 0
  !$omp parallel do reduction(+:total)
  do value = 1, 1000000
    total = total + value
  end do
  !$omp end parallel do
  print *, 'reduction:', total
end subroutine
call reduction_demo()

In [ ]:
subroutine sections_demo()
  integer :: first, second
  first = 0; second = 0
  !$omp parallel sections
  !$omp section
  first = 20
  !$omp section
  second = 22
  !$omp end parallel sections
  print *, 'sections:', first + second
end subroutine
call sections_demo()

In [ ]:
subroutine tasks_demo()
  integer :: values(4), task_index
  values = 0
  !$omp parallel
  !$omp single
  do task_index = 1, 4
    !$omp task firstprivate(task_index) shared(values)
    values(task_index) = task_index * task_index
    !$omp end task
  end do
  !$omp taskwait
  !$omp end single
  !$omp end parallel
  print *, 'tasks:', sum(values)
end subroutine
call tasks_demo()

The `%%mlir` magic lets us inspect the lowered OpenMP program without executing it.

In [ ]:
%%mlir
subroutine openmp_sum(result)
  integer(kind=8), intent(out) :: result
  integer(kind=8) :: i
  result = 0
  !$omp parallel do reduction(+:result)
  do i = 1, 1000
    result = result + i
  end do
  !$omp end parallel do
end subroutine